In [ ]:
import netCDF4 as nc
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import geopandas as gpd
import xarray as xr
import rioxarray
from shapely.geometry import mapping
import rasterio
from rasterio.enums import Resampling

In [ ]:
# 打开包含风速、温度和太阳辐射的 NetCDF 文件
# 文件名: "6windtempsolar201920.nc"
slope = xr.open_dataset(r"F:\Workspace\DATA_DEM\Slope_map_1km.tif")["band_data"][0, :, :] #作为投影参考
data = xr.open_dataset(r"H:\ClimateData\2005.nc")

# 提取 100 米高度的东向风速 (u 分量) 和北向风速 (v 分量)
# 这里使用 8760 索引开始，表示从第二年的数据开始（假设每小时记录一次）
i=0
u100 = data["u100"][0:8760, :, :]
v100 = data["v100"][0:8760, :, :]
u100 = u100.rio.write_crs(slope.rio.crs) #重新进行投影
v100 = v100.rio.write_crs(slope.rio.crs) #重新进行投影

In [ ]:
slope = xr.open_dataset(r"F:\Workspace\DATA_DEM\Slope_map_025d.tif")["band_data"][0, :, :]
slope_1km = xr.open_dataset(r"F:\Workspace\DATA_DEM\Slope_map_1km.tif")["band_data"][0, :, :]
lucc = xr.open_dataset(r"F:\Workspace\DATA_Landuse\World_landuse_025d.tif")["band_data"][0, :, :]
lucc_1km = xr.open_dataset(r"F:\Workspace\DATA_Landuse\World_landuse_1km.tif")["band_data"][0, :, :]
elevation = xr.open_dataset(r"F:\Workspace\DATA_DEM\merged_global_map_025d.tif")["band_data"][0, :, :]
elevation_1km = xr.open_dataset(r"F:\Workspace\DATA_DEM\merged_global_map.tif")["band_data"][0, :, :]
protected = xr.open_dataset(r"F:\Workspace\DATA_Protected\World_new.tif")["band_data"][0, :, :].rio.reproject_match(u100, resampling=Resampling.mode)
protected_1km = xr.open_dataset(r"F:\Workspace\DATA_Protected\Protected_1km.tif")["band_data"][0, :, :]
classification_df = pd.read_csv("land_use_classification.csv")
lucc_Wind_weights = dict(zip(classification_df['Flag Value'], classification_df['Onshore wind-Base']))
lucc_Solar_weights = dict(zip(classification_df['Flag Value'], classification_df['Solar-Base']))
lucc_cf = dict(zip(classification_df['Flag Value'], classification_df['CF']))
lucc_cf_noWater = dict(zip(classification_df['Flag Value'], classification_df['CF_noWater']))
slope_1km = slope_1km.rename({'y': 'latitude', 'x': 'longitude'})

In [ ]:
def Wind_slope_process(x):
    #conditions = [(x >= 0) | (x <= 15), (x > 15)] #Conservative
    conditions = [(x >= 0) | (x <= 20), (x > 20)] #Base
    #conditions = [(x >= 0) | (x <= 25), (x > 25)] #Open
    slopes = [1, 0]
    return np.select(conditions, slopes, default=np.nan)
wind_slope = Wind_slope_process(slope)

def Solar_slope_process(x):
    #conditions = [(x >= 0) | (x <= 3), (x > 3)] #Conservative
    conditions = [(x >= 0) | (x <= 5), (x > 5)] #Base
    #conditions = [(x >= 0) | (x <= 7), (x > 7)] #Open
    slopes = [1, 0]
    return np.select(conditions, slopes, default=np.nan)
solar_slope = Solar_slope_process(slope)

def elevation_process(x):
    conditions = [x<=3000, x>3000]
    elevations = [1, 0]
    return np.select(conditions, elevations, default=np.nan)
elevation = elevation_process(elevation)

def protected_process(x):
    conditions = [x==0, x>0]
    protected = [1, 0]
    return np.select(conditions, protected, default=np.nan)
protected = protected_process(protected)

def CF_process_wind(x):
    conditions = [(x < 3) | (x > 25), (x < 5), (x < 12), (x <= 25)]
    CFs = [0, 125*(x+1e-7-3)/2500, (250 + 2250/7 * (x-5))/2500, 1]
    return np.select(conditions, CFs, default=np.nan)

def pv_power(T, S, wind10):  # 光伏小时容量因子计算函数
    # T: 环境温度 (摄氏度)
    # S: 太阳辐射量 (单位：W/m²)
    # wind10: 10米高度的风速 (单位：m/s)
    
    # 计算光伏组件温度 (T_p)
    # T_p 表示组件的实际温度，它受环境温度、太阳辐射量以及风速的影响
    T_p = 4.3 + 0.943 * T + 0.028 * S / 3600 - 1.528 * wind10

    # 计算温度对光伏组件效率的影响 (TEM)
    # 组件温度高于25度会降低效率，每升高1度，效率降低0.5%
    TEM = (1 - 0.005 * (T_p - 25))

    # 系统效率 (SYS)
    # 包括逆变器、线缆等系统损耗，假设为85%
    SYS = 0.85

    # 返回光伏小时容量因子
    return TEM * SYS


def pv_power1(T, S, wind10):  # 光伏小时容量因子计算函数（改进版）
    # T: 环境温度 (摄氏度)
    # S: 太阳辐射量 (单位：W/m²)
    # wind100: 10米高度的风速 (单位：m/s)

    # 计算光伏组件温度 (T_p)
    # 与前一个函数相同，计算组件温度
    T_p = 4.3 + 0.943 * T + 0.028 * S / 3600 - 1.528 * wind10

    # 计算温度对光伏组件效率的影响 (TEM)
    TEM = (1 - 0.005 * (T_p - 25))

    # 系统效率 (SYS)
    SYS = 0.85

    # 返回光伏小时容量因子
    # 在这里，计算的单位从 W 转换为 kWh，S/1000 将 S 单位从 W/m² 转换为 kW/m²，除以 3600 以获得小时数的因子
    return S / 1000 / 3600 * TEM * SYS



In [ ]:
import xarray as xr

# 定义年份范围
years =list(range(2004, 2024))  # 包含 2013 到 2023 年

# 定义经度分段数
num_lon_segments = 24

# 逐段进行计算并保存结果
for i in range(0, 1440, 60):
    print(f"Processing longitude segment: {i}")
    
    # 初始化累计和变量
    CF_wind_sum = 0
    CF_pv_sum = 0
    ssrd_adjust_sum = 0

    # 初始化统一的时间坐标
    time_coords = None

    # 逐年读取数据并进行累加
    for year in years[::-1]:
        # 获取文件路径
        if year < 2008:
            file_path_template = r"D:\Workspace\{}.nc"
        elif year < 2016:
            file_path_template = r"E:\Workspace\{}.nc"
        else:
            file_path_template = r"F:\Workspace\{}.nc"
        
        file_path = file_path_template.format(year)
        
        # 打开每年的 NetCDF 文件
        with xr.open_dataset(file_path) as data:
            print(f"Processing year: {year}")
            # 计算每个纬度分段的起始和结束索引

            Cover_wind = xr.open_dataset(f"E:/Workspace/Data_temp/wind_cover_chunk_{i}.nc")
            Cover_solar = xr.open_dataset(f"E:/Workspace/Data_temp/pv_cover_chunk_{i}.nc")
            
            # 提取所需的变量并截取时间范围和纬度范围
            u100 = data["u100"][0:8760, :, i:i+60]
            v100 = data["v100"][0:8760, :, i:i+60]
            u10 = data["u10"][0:8760, :,i:i+60]
            v10 = data["v10"][0:8760, :, i:i+60]
            ssrd = data["ssrd"][0:8760, :, i:i+60]
            t2m = data["t2m"][0:8760, :, i:i+60]
            
            wind_100 = np.sqrt(u100**2 + v100**2)
            wind_10 = np.sqrt(u10**2 + v10**2)
            CF_wind = CF_process_wind(wind_100)* np.array(Cover_wind.to_array())[0]
            
            t2m.values = t2m.values - 273.15
            ssrd.values = ssrd.values
            CF_pv = np.array(xr.apply_ufunc(pv_power1, t2m, ssrd, wind_10)) * np.array(Cover_solar.to_array())[0]
            alpha_pv = xr.apply_ufunc(pv_power, t2m, ssrd, wind_10)
            ssrd_adjust =ssrd.values*np.array(alpha_pv)

            # 累加每年的数据
            if CF_wind is None:
                CF_wind_sum = CF_wind
                CF_pv_sum = CF_pv
                ssrd_adjust_sum = ssrd_adjust

            else:
                CF_wind_sum += CF_wind
                CF_pv_sum += CF_pv
                ssrd_adjust_sum += ssrd_adjust
                
    # 计算平均值
    num_years = len(years)
    CF_wind_avg = CF_wind_sum / num_years
    CF_pv_avg = CF_pv_sum / num_years
    ssrd_adjust_avg = ssrd_adjust_sum / num_years

    # 创建一个数据集来保存平均结果
    avg_data = xr.Dataset(
        {
            "CF_wind_avg": (["valid_time", "latitude", "longitude"], CF_wind_avg.astype(np.float32)),
            "CF_pv_avg": (["valid_time", "latitude", "longitude"], CF_pv_avg.astype(np.float32)),
            "ssrd_adjust_avg": (["valid_time", "latitude", "longitude"], ssrd_adjust_avg.astype(np.float32)),
        },
        coords={
            "valid_time": u100.valid_time,
            "latitude": u100.latitude,
            "longitude": u100.longitude
        }
    )

    # 打印平均数据的信息
    print(avg_data)

    # 保存平均结果为新的 NetCDF 文件
    avg_data.to_netcdf(rf"D:\Workspace\average_2004_2023_segment_{i}.nc")


In [ ]:
import xarray as xr
import numpy as np

# 定义年份范围
years =list(range(2021,2022))  # 包含 2013 到 2023 年

# 定义经度分段数
num_lon_segments = 24

# 逐段进行计算并保存结果
for i in range(0, 1440, 60):
    print(f"Processing longitude segment: {i}")
    
    # 初始化累计和变量
    CF_wind_sum = 0
    CF_pv_sum = 0
    ssrd_adjust_sum = 0

    # 初始化统一的时间坐标
    time_coords = None

    # 逐年读取数据并进行累加
    for year in years[::-1]:

        file_path_template = r"H:\ClimateData\{}.nc"       
        
        file_path = file_path_template.format(year)
        
        # 打开每年的 NetCDF 文件
        with xr.open_dataset(file_path) as data:
            print(f"Processing year: {year}")
            # 计算每个纬度分段的起始和结束索引

            #Cover_wind = xr.open_dataset(f"E:/Workspace/Data_temp/wind_cover_chunk_{i}.nc")
            #Cover_solar = xr.open_dataset(f"E:/Workspace/Data_temp/pv_cover_chunk_{i}.nc")
            
            # 提取所需的变量并截取时间范围和纬度范围
            u100 = data["u100"][0:8760, :, i:i+60]
            v100 = data["v100"][0:8760, :, i:i+60]
            u10 = data["u10"][0:8760, :,i:i+60]
            v10 = data["v10"][0:8760, :, i:i+60]
            ssrd = data["ssrd"][0:8760, :, i:i+60]
            t2m = data["t2m"][0:8760, :, i:i+60]
            
            wind_100 = np.sqrt(u100**2 + v100**2)
            wind_10 = np.sqrt(u10**2 + v10**2)
            #CF_wind = CF_process_wind(wind_100)* np.array(Cover_wind.to_array())[0]
            CF_wind = CF_process_wind(wind_100)
            
            t2m.values = t2m.values - 273.15
            ssrd.values = ssrd.values
            #CF_pv = np.array(xr.apply_ufunc(pv_power1, t2m, ssrd, wind_10)) * np.array(Cover_solar.to_array())[0]
            CF_pv = np.array(xr.apply_ufunc(pv_power1, t2m, ssrd, wind_10))
            alpha_pv = xr.apply_ufunc(pv_power, t2m, ssrd, wind_10)
            ssrd_adjust =ssrd.values*np.array(alpha_pv)

            # 累加每年的数据
            if CF_wind is None:
                CF_wind_sum = CF_wind
                CF_pv_sum = CF_pv
                ssrd_adjust_sum = ssrd_adjust

            else:
                CF_wind_sum += CF_wind
                CF_pv_sum += CF_pv
                ssrd_adjust_sum += ssrd_adjust
                
    # 计算平均值
    num_years = len(years)
    CF_wind_avg = CF_wind_sum / num_years
    CF_pv_avg = CF_pv_sum / num_years
    ssrd_adjust_avg = ssrd_adjust_sum / num_years

    # 创建一个数据集来保存平均结果
    avg_data = xr.Dataset(
        {
            "CF_wind_avg": (["valid_time", "latitude", "longitude"], CF_wind_avg.astype(np.float32)),
            "CF_pv_avg": (["valid_time", "latitude", "longitude"], CF_pv_avg.astype(np.float32)),
            "ssrd_adjust_avg": (["valid_time", "latitude", "longitude"], ssrd_adjust_avg.astype(np.float32)),
        },
        coords={
            "valid_time": u100.valid_time,
            "latitude": u100.latitude,
            "longitude": u100.longitude
        }
    )

    # 打印平均数据的信息
    print(avg_data)

    # 保存平均结果为新的 NetCDF 文件
    avg_data.to_netcdf(rf"D:\Workspace\average_2021_segment_{i}.nc")


In [ ]:
import xarray as xr
data = xr.open_dataset(f"D:/Workspace/average_2023_segment_{0 * 60}.nc")

In [ ]:
data

In [ ]:
import numpy as np
np.array(data.to_array())

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import TwoSlopeNorm
plt.imshow(np.array(data.to_array())[0,0,:,:])